In [ ]:
# Imports 
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Load 
SEED = 123
df = pd.read_csv("../Dati/Processed/dataset_processed_quantile1_sentences.csv")
X, y = df["text_bert"], df["binary_label"].values
cv = [(np.where(df.fold != k)[0], np.where(df.fold == k)[0]) for k in sorted(df.fold.unique())]

In [ ]:
# Pipeline + grid search 
# linear-baseline family (like logreg) but max-margin; vectorizer refit per fold -> no leakage
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=0.025, sublinear_tf=True)),
    ("clf", LinearSVC(class_weight="balanced", random_state=SEED, max_iter=8000)),
])
gs = GridSearchCV(pipe, {"clf__C": [0.3, 0.5, 1.0, 2.0]}, scoring="f1_macro", cv=cv, n_jobs=-1).fit(X, y)
print(gs.best_params_, round(gs.best_score_, 3))

{'clf__C': 1.0} 0.661


In [ ]:
# Out-of-fold predictions
oof = cross_val_predict(gs.best_estimator_, X, y, cv=cv)

In [ ]:
# Classification report + confusion matrix 
print(classification_report(y, oof, digits=3))
print(confusion_matrix(y, oof))

              precision    recall  f1-score   support

           0      0.589     0.476     0.527       208
           1      0.761     0.834     0.796       416

    accuracy                          0.715       624
   macro avg      0.675     0.655     0.661       624
weighted avg      0.704     0.715     0.706       624

[[ 99 109]
 [ 69 347]]
